# Daily Market Statistics - By Indranil

In [0]:
%pip install -q weasyprint

## (Step 1): Prepare & Run Python Scripts:

In [0]:
import pandas as pd
from datetime import datetime as dt, timedelta
import altair as alt
import mplfinance as mpf
import plotly.graph_objects as go

from utility.trading.nse_api import (get_nse_market_status_daily, 
                                     get_nse_index_daily,
                                     get_nse_india_vix,
                                     get_nse_date)

# # date functionality
# current_date_time =  dt.now().strftime('%d-%m-%Y %H:%M:%S')
# current_date = dt.now().strftime('%d-%m-%Y')
# one_yr_prev_date = (dt.now() - timedelta(days=365)).strftime('%d-%m-%Y')

# Get current date and market date
nse_current_date = get_nse_date()

# Generate nse daily status
nse_market_status_daily_df = get_nse_market_status_daily()
nse_index_daily_df = get_nse_index_daily()

# Generate nifty india vix
nse_india_vix_yearly_df = get_nse_india_vix()  # default parameter will generate data for one year
# nse_india_vix_yearly_chart_df = nse_india_vix_yearly_df.copy()
# nse_india_vix_yearly_chart_df['datetime'] = nse_india_vix_yearly_chart_df['date'].apply(lambda x: dt.strptime(x, '%d-%b-%Y'))  # .strftime('%Y-%m-%d %H:%M:%S'))
# nse_india_vix_yearly_chart_df.set_index('datetime', inplace=True)
# nse_india_vix_yearly_chart_df.drop(columns=['date'], inplace=True)

# Generate Chart Figure
nse_index_daily_chart_fig = alt.Chart(nse_index_daily_df).mark_bar().encode(
    x='index_name:N',  # specify nominal data
    y='perchange:Q',  # specify quantitative data
    color=alt.Color('perchange', scale=alt.Scale(scheme='darkred'))
).properties(width=700, height=300)


# yellow green	#9ACD32	(154,205,50)
# tomato	#FF6347	(255,99,71)
nse_index_daily_color = pd.DataFrame()
nse_index_daily_color['color'] = nse_index_daily_df['perchange'].apply(lambda x: 'rgb(154, 205, 50)' if x > 0 else 'rgb(255, 99, 71)')
nse_index_daily_tabular_chart_fig = go.Figure(data=[go.Table(
  header=dict(
    values=['<b>'+i+'</b>' if i == 'perchange' else i for i in nse_index_daily_df.columns],
    line_color='darkslategray', fill_color='white',
    align='center', font=dict(color='black', size=12)
  ),
  cells=dict(
    values=[nse_index_daily_df[col] for col in nse_index_daily_df.columns],
    line_color=['darkslategray'], fill_color=[nse_index_daily_color.color],
    align='left', font=dict(color='black', size=10)
  ))
])
# nse_index_daily_tabular_chart_fig.update_layout(width=950, height=650)
nse_index_daily_tabular_chart_fig.update_layout(width=950, height=640)

nse_india_vix_yearly_chart_fig = go.Figure(data=[go.Candlestick(x=nse_india_vix_yearly_df['date'],
                open=nse_india_vix_yearly_df['open'],
                high=nse_india_vix_yearly_df['high'],
                low=nse_india_vix_yearly_df['low'],
                close=nse_india_vix_yearly_df['close'])])

In [0]:
nse_index_daily_tabular_chart_fig.update_layout(width=950, height=640)

## (Step 2): Generate Dynamic HTML Content

In [0]:
dynamic_html = f"""
    <h1>Daily Market Statistics As of <u>{nse_current_date}</u></h1>

    <hr>
    <p>
        <label>Indian Stock Exchanges: </label>
        <a href='https://www.nseindia.com' target='_blank' rel='noopener noreferrer'>NSE</a> | <a href='https://www.bseindia.com/' target='_blank' rel='noopener noreferrer'>BSE</a>
    </p>

    <hr>
    <p>
        <label>Other Market Analysis Tools: </label>
        <ul>
            <li><a href='#'>Screener</a></li>
            <li><a href='#'>Value Research</a></li>
            <li><a href='#'>TradingView</a></li>
        </ul>
    </p>

    <hr>
    <div class='side-by-side'>
        <div>
            <p>Nifty 50: (status:<b>{nse_market_status_daily_df.loc[:, 'nifty50_status'].values[0]}</b>)</p>
            <p>{nse_market_status_daily_df.loc[:,['nifty50_close', 'nifty50_change', 'nifty50_perchange']].rename(columns={'nifty50_close': 'Close', 'nifty50_change': 'Change', 'nifty50_perchange': 'PerChange'}).to_html(index=False)}</p>
        </div>
        <div>
            <p>Gift Nifty 50:</p>
            <p>{nse_market_status_daily_df.loc[:,['giftnifty_close', 'giftnifty_change', 'giftnifty_perchange']].rename(columns={'giftnifty_close': 'Close', 'giftnifty_change': 'Change', 'giftnifty_perchange': 'PerChange'}).to_html(index=False)}</p>
        </div>
        <div>
            <p>Market Cap:</p>
            <p>{nse_market_status_daily_df.loc[:,['marketcapin_trdollars', 'marketcapin_laccr_rupees']].rename(columns={'marketcapin_trdollars': 'Trillion $', 'marketcapin_laccr_rupees': 'Lac Cr ₹'}).to_html(index=False)}</p>
        </div>
    </div>
    
    <hr>
    <h2'>Nifty Index</h2>
    <hr>
    {nse_index_daily_tabular_chart_fig.to_html()}

    <hr>
    <h2'>Nifty India Vix</h2>
    <hr>
    {nse_india_vix_yearly_chart_fig.to_html()}
    <br>
"""


### Add Customized Style with Dynamic HTML

In [0]:
main_wrapper_html_with_style = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8" />   <!--It is necessary to use the UTF-8 encoding with plotly graphics to get e.g. negative signs to render correctly -->

        <meta name="viewport" content="width=device-width, initial-scale=1.0" />
 
        <style>
            body, p {{
                padding: 0px 10px;
            }}
            .side-by-side {{
                display: flex;
            }}
            .side-by-side > div {{
                width: 33%;
                border: 1px solid;
            }}
        </style>
    </head>
    <body>
        {dynamic_html}
    </body>
    </html>
"""

## (Step 3): Analysis The HTML Content With Gemini

In [0]:
GEMINI_API_KEY = dbutils.secrets.get(scope="Databricks_Scope", key="GEMINI_API_KEY")

## (Step 4): Final HTML Output

In [0]:
displayHTML(main_wrapper_html_with_style)

## (Step 5): Generate The PDF from Dynamic HTML Content

In [0]:
# from weasyprint import HTML

# HTML(string=main_wrapper_html_with_style).write_pdf('//Workspace/Users/infoeasycloudapi@gmail.com/AlgoTrading-With-AIAssistance/output/nse_market_status_daily.pdf')

## Sample Chart

In [0]:
import altair as alt
from altair.datasets import data

source = data.population.url

alt.Chart(source).mark_boxplot(extent='min-max').encode(
    x='age:O',
    y='people:Q'
)